In [2]:
import pandas as pd

df = pd.read_csv(
    "../reports/gate3_evaluation/validation_rows.csv"
)

print(df.shape)
print(df.columns.tolist())
df.head()

(49620, 10)
['baseline', 'chunk_id', 'title', 'context_group_id', 'category', 'score', 'threshold', 'detected', 'target', 'mask']


,baseline,chunk_id,title,context_group_id,category,score,threshold,detected,target,mask
0,keyword,"ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...","ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...",c757bb1d6007551134d3831a8e3ee3ef1491c59513844f...,Anti-Assignment,0.0,2.0,False,0,0
1,keyword,"ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...","ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...",c757bb1d6007551134d3831a8e3ee3ef1491c59513844f...,Cap On Liability,0.0,2.0,False,0,0
2,keyword,"ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...","ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...",c757bb1d6007551134d3831a8e3ee3ef1491c59513844f...,Change Of Control,0.0,2.0,False,0,0
3,keyword,"ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...","ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...",c757bb1d6007551134d3831a8e3ee3ef1491c59513844f...,Exclusivity,1.0,3.0,False,1,0
4,keyword,"ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...","ADIANUTRITION,INC_04_01_2005-EX-10.D2-RESELLER...",c757bb1d6007551134d3831a8e3ee3ef1491c59513844f...,Expiration Date,1.0,3.0,False,0,0


In [3]:
usable = df[df["mask"] == 0].copy()

print("All rows:", len(df))
print("Usable rows:", len(usable))

All rows: 49620
Usable rows: 49292


In [5]:
def calculate_metrics(data):
    tp = (
        (data["target"] == 1)
        & (data["detected"] == True)
    ).sum()

    fp = (
        (data["target"] == 0)
        & (data["detected"] == True)
    ).sum()

    fn = (
        (data["target"] == 1)
        & (data["detected"] == False)
    ).sum()

    tn = (
        (data["target"] == 0)
        & (data["detected"] == False)
    ).sum()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [6]:
for baseline in ["keyword", "tfidf"]:
    baseline_data = usable[
        usable["baseline"] == baseline
    ]

    result = calculate_metrics(
        baseline_data
    )

    print("\n", baseline)
    print(result)


 keyword
{'TP': np.int64(893), 'FP': np.int64(352), 'FN': np.int64(449), 'TN': np.int64(22952), 'precision': np.float64(0.7172690763052209), 'recall': np.float64(0.665424739195231), 'f1': np.float64(0.6903749516814843)}

 tfidf
{'TP': np.int64(987), 'FP': np.int64(308), 'FN': np.int64(355), 'TN': np.int64(22996), 'precision': np.float64(0.7621621621621621), 'recall': np.float64(0.7354694485842027), 'f1': np.float64(0.7485779294653016)}


In [7]:
rows = []

for baseline in ["keyword", "tfidf"]:

    for category in usable["category"].unique():

        category_data = usable[
            (usable["baseline"] == baseline)
            & (usable["category"] == category)
        ]

        result = calculate_metrics(
            category_data
        )

        rows.append({
            "baseline": baseline,
            "category": category,
            **result
        })

category_results = pd.DataFrame(rows)

category_results

,baseline,category,TP,FP,FN,TN,precision,recall,f1
0,keyword,Anti-Assignment,169,44,48,2187,0.793427,0.778802,0.786047
1,keyword,Cap On Liability,138,19,91,2198,0.878981,0.602620,0.715026
2,keyword,Change Of Control,49,97,24,2302,0.335616,0.671233,0.447489
3,keyword,Exclusivity,74,67,45,2270,0.524823,0.621849,0.569231
4,keyword,Expiration Date,115,22,77,2251,0.839416,0.598958,0.699088
5,keyword,License Grant,145,27,62,2224,0.843023,0.700483,0.765172
6,keyword,Non-Compete,39,52,33,2355,0.428571,0.541667,0.478528
7,keyword,Non-Transferable License,77,3,12,2384,0.962500,0.865169,0.911243
8,keyword,Renewal Term,55,14,23,2381,0.797101,0.705128,0.748299
9,keyword,Uncapped Liability,32,7,34,2400,0.820513,0.484848,0.609524


In [8]:
macro_results = (
    category_results
    .groupby("baseline")["f1"]
    .mean()
)

print(macro_results)

baseline
keyword    0.672964
tfidf      0.711664
Name: f1, dtype: float64
